# Data initialization (model-ready)

Loads `processed_data/f2_combined_raw.csv` and derives consistent, model-ready columns using the proxy guidance in `raw_data/README.md`. Each value column `F2XXX` is paired with an imputation flag `XF2XXX`; we propagate per-row data-quality flags so downstream models can filter or weight imputed observations.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RAW = ROOT / "raw_data"
PROCESSED = ROOT / "processed_data"

combined_path = PROCESSED / "f2_combined_raw.csv"
combined_path

In [ ]:
df = pd.read_csv(combined_path, low_memory=False)
df.shape, df.columns.tolist()

In [ ]:
df.head(3)

In [ ]:
NUMERIC_COLS = [
    "F2D01", "F2D16",
    "F2A02", "F2A03",
    "F2B02", "F2B04",
    "F2A04", "F2A05B",
    "F2I03", "F2I05", "F2I07",
    "F2H01", "F2H02",
]
for c in NUMERIC_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["UNITID"] = pd.to_numeric(df["UNITID"], errors="coerce").astype("Int64")
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

df[["UNITID", "year"]].isna().sum().to_dict()

In [ ]:
# IPEDS imputation flag values (per F2 documentation):
#   R = Reported, A = Analyst-corrected (treat as reported)
#   C = Carried forward, P = Prior-year imputation, Z = Zero-imputed, N = Not applicable
REPORTED_FLAGS = {"R", "A"}

def is_reported(flag_col: str) -> pd.Series:
    if flag_col not in df.columns:
        return pd.Series(False, index=df.index)
    return df[flag_col].isin(REPORTED_FLAGS)

In [ ]:
# Model-ready proxies (raw_data/README.md guidance):
# - total_expenses        = F2I07 if present else F2B02
# - change_in_net_assets  = F2I03 if present else F2B04
# - expendable_net_assets = F2I05 if present else (F2A04 + F2A05B)
df["total_expenses"] = df["F2I07"].where(df["F2I07"].notna(), df["F2B02"])
df["change_in_net_assets"] = df["F2I03"].where(df["F2I03"].notna(), df["F2B04"])
df["expendable_net_assets"] = df["F2I05"].where(df["F2I05"].notna(), df["F2A04"] + df["F2A05B"])

# Track which source field fed each proxy (so the quality flag points at the right XF2*)
df["_src_total_expenses"] = np.where(df["F2I07"].notna(), "XF2I07", "XF2B02")
df["_src_change_in_net_assets"] = np.where(df["F2I03"].notna(), "XF2I03", "XF2B04")
df["_src_expendable_net_assets"] = np.where(df["F2I05"].notna(), "XF2I05", "XF2A04+XF2A05B")

df["endowment_boy"] = df.get("F2H01")
df["endowment_eoy"] = df.get("F2H02")

In [ ]:
# Per-row data-quality flag: for each model-ready measure, was the underlying IPEDS value reported?
df["q_F2D01"] = is_reported("XF2D01")
df["q_F2D16"] = is_reported("XF2D16")
df["q_F2A02"] = is_reported("XF2A02")
df["q_F2A03"] = is_reported("XF2A03")
df["q_total_expenses"] = np.where(
    df["_src_total_expenses"] == "XF2I07", is_reported("XF2I07"), is_reported("XF2B02")
)
df["q_change_in_net_assets"] = np.where(
    df["_src_change_in_net_assets"] == "XF2I03", is_reported("XF2I03"), is_reported("XF2B04")
)
df["q_expendable_net_assets"] = np.where(
    df["_src_expendable_net_assets"] == "XF2I05",
    is_reported("XF2I05"),
    is_reported("XF2A04") & is_reported("XF2A05B"),
)
df["q_endowment_eoy"] = is_reported("XF2H02")

QUALITY_COLS = [
    "q_F2D01", "q_F2D16", "q_F2A02", "q_F2A03",
    "q_total_expenses", "q_change_in_net_assets",
    "q_expendable_net_assets", "q_endowment_eoy",
]
df["reported_field_count"] = df[QUALITY_COLS].sum(axis=1).astype("Int64")
df["all_fields_reported"] = df[QUALITY_COLS].all(axis=1)

df["reported_field_count"].value_counts().sort_index()

In [ ]:
# Convenience ratios commonly used for financial health modeling.
df["assets_to_liabilities"] = df["F2A02"] / df["F2A03"].replace({0: np.nan})
df["operating_margin"] = df["change_in_net_assets"] / df["F2D16"].replace({0: np.nan})
df["expenses_to_revenue"] = df["total_expenses"] / df["F2D16"].replace({0: np.nan})

df[[
    "total_expenses", "change_in_net_assets", "expendable_net_assets",
    "assets_to_liabilities", "operating_margin", "expenses_to_revenue",
]].describe().T

In [ ]:
FEATURE_COLS = [
    "UNITID", "year",
    "F2D01", "F2D16", "F2A02", "F2A03",
    "total_expenses", "change_in_net_assets", "expendable_net_assets",
    "assets_to_liabilities", "operating_margin", "expenses_to_revenue",
    "endowment_boy", "endowment_eoy",
    *QUALITY_COLS,
    "reported_field_count", "all_fields_reported",
]

model_df = df[FEATURE_COLS].copy()
model_df = model_df.dropna(subset=["UNITID", "year"]).sort_values(["UNITID", "year"], kind="mergesort")
model_df.shape, model_df.head(3)

In [ ]:
out_path = PROCESSED / "f2_model_ready.csv"
model_df.to_csv(out_path, index=False)
out_path